# So sánh dataset đã tạo với paper DeepSMOTE (Google Colab)

Notebook này **chỉ đọc/tạo dataset rồi so sánh** — không train autoencoder, không SMOTE, không GAN.
Dùng lại các hàm trong `src/visualization/report.py` và `src/preprocessing/create_imbalanced_dataset.py`, không viết lại logic.

**Cách chạy:** `Runtime > Run all`. Cell đầu tiên sẽ tự clone repo về `/content/GAN` và cài dependency. Nếu dataset chưa có, cell ở mục "Chọn dataset và seed" sẽ tự chạy preprocessing (tải dataset gốc + sample) trước khi so sánh.

Sửa `REPO_URL` bên dưới nếu bạn fork repo sang tài khoản khác.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ColorOfDreams/GAN.git"

if IN_COLAB:
    REPO_DIR = Path("/content/GAN")
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)
    # torch/torchvision đã có sẵn trên Colab và thỏa điều kiện >=, pip sẽ bỏ qua, không cài lại bản to
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    REPO_DIR = Path.cwd()

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Thư mục project:", REPO_DIR)
print("Chạy trên Colab:", IN_COLAB)

## (Tùy chọn) Lưu dữ liệu qua Google Drive

Colab xóa hết file sau khi hết phiên, nên mỗi lần mở lại phải tải MNIST/CIFAR-10/... và sample lại từ đầu. Nếu muốn giữ `data/processed/` (và `reports/`) giữa các lần chạy, đặt `USE_DRIVE = True` ở cell dưới rồi chạy — notebook sẽ mount Drive và trỏ `data/processed`, `reports/` vào đó qua symlink. Bỏ qua bước này nếu chỉ chạy thử nhanh.

In [ ]:
USE_DRIVE = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    drive_root = Path("/content/drive/MyDrive/GAN_deepsmote_data")
    for name in ("processed", "raw"):
        (drive_root / name).mkdir(parents=True, exist_ok=True)
        local_path = REPO_DIR / "data" / name
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if local_path.is_symlink() or local_path.exists():
            continue
        os.symlink(drive_root / name, local_path, target_is_directory=True)
    print("Đã trỏ data/processed và data/raw vào Google Drive:", drive_root)

In [ ]:
from preprocessing.config import DATASET_CONFIGS
from preprocessing.create_imbalanced_dataset import create_imbalanced_dataset
from visualization.report import (
    load_processed_dataset,
    compare_with_deepsmote,
    comparison_all_match,
    format_comparison_table,
    plot_class_distribution,
    plot_sample_grid,
)

## Chọn dataset và seed

`DATASET` là một trong: `mnist`, `fashion_mnist`, `cifar10`, `svhn`, `celeba`. Nếu dataset này chưa được tạo trong `data/processed/`, cell dưới sẽ tự chạy `create_imbalanced_dataset` (tải dataset gốc + sample theo đúng số liệu paper DeepSMOTE) trước khi load. CelebA có thể lỗi vì Google Drive giới hạn lượt tải — xem README nếu gặp lỗi này.

In [ ]:
DATASET = "mnist"
SEED = 42

try:
    ds = load_processed_dataset(DATASET, SEED)
except FileNotFoundError:
    print(f"Chưa có dataset '{DATASET}' (seed={SEED}), đang tạo...")
    create_imbalanced_dataset(DATASET, seed=SEED)
    ds = load_processed_dataset(DATASET, SEED)

print(f"Đã load {ds.cfg.display_name}: train={len(ds.train_y)} mẫu, test={len(ds.test_y)} mẫu")

## Bảng so sánh số mẫu mỗi class: paper DeepSMOTE vs. dataset đã tạo

In [ ]:
rows = compare_with_deepsmote(ds)
print(format_comparison_table(rows))
print()
print("Giống paper DeepSMOTE:", "CÓ" if comparison_all_match(rows) else "KHÔNG")

## Biểu đồ phân bố class (paper vs. thực tế)

In [ ]:
fig = plot_class_distribution(ds, rows)

## Lưới ảnh mẫu theo từng class

In [ ]:
fig = plot_sample_grid(ds, samples_per_class=6, seed=SEED)

## So sánh nhanh các dataset đã tạo

Chỉ kiểm tra dataset **đã có sẵn** trong `data/processed/` (không tự tạo hết, vì CIFAR-10/SVHN/CelebA tải khá lâu). Muốn thêm dataset nào vào bảng này, chạy cell "Chọn dataset và seed" ở trên với `DATASET` tương ứng trước.

In [ ]:
for key in DATASET_CONFIGS:
    try:
        ds_i = load_processed_dataset(key, SEED)
    except FileNotFoundError as exc:
        print(f"{key:16s} -> chưa tạo")
        continue
    rows_i = compare_with_deepsmote(ds_i)
    status = "CÓ" if comparison_all_match(rows_i) else "KHÔNG"
    print(f"{key:16s} -> giống paper DeepSMOTE: {status}")